## Reconstruct corrupt images

Task is to take a corrupted 32x32 image and predict the clean one. Scored on MSE.

Submitting the corrupt images unchanged gives about 500. Roughly half the total
error comes from the ~14% of images that are rotated, so the model predicts the
rotation angle and undoes it before denoising.

## Data

In [ ]:
import os
import glob
import csv
import numpy as np
from PIL import Image

# find the competition folder
root = '/kaggle/input/'
if os.environ.get('ROOT'):
    root = os.environ['ROOT']
else:
    for d in sorted(glob.glob('/kaggle/input/*/')):
        if os.path.exists(d + 'train.csv') and os.path.isdir(d + 'test_corrupt'):
            root = d
    for d in sorted(glob.glob('/kaggle/input/*/*/')):
        if os.path.exists(d + 'train.csv') and os.path.isdir(d + 'test_corrupt'):
            root = d
out = '/kaggle/working/'
if os.environ.get('OUT'):
    out = os.environ['OUT']
print(root)
print(out)

In [ ]:
# load data
f = open(root + 'train.csv')
df = []
for r in csv.DictReader(f):
    df.append(r)
f.close()
print(len(df))
print(df[0])

In [ ]:
K_np = np.zeros((len(df), 32, 32, 3), np.uint8)
C_np = np.zeros((len(df), 32, 32, 3), np.uint8)
for i in range(len(df)):
    im = Image.open(root + 'train_clean/' + df[i]['clean_filename']).convert('RGB')
    K_np[i] = np.asarray(im)
    im2 = Image.open(root + 'train_corrupt/' + df[i]['corrupt_filename']).convert('RGB')
    C_np[i] = np.asarray(im2)
print(K_np.shape)
print(C_np.shape)

In [ ]:
f2 = open(root + 'sample_submission.csv')
temp = []
for r in csv.reader(f2):
    temp.append(r[0])
f2.close()
test_ids = temp[1:]
print(len(test_ids))
print(test_ids[0])

In [ ]:
T_np = np.zeros((len(test_ids), 32, 32, 3), np.uint8)
for i in range(len(test_ids)):
    im3 = Image.open(root + 'test_corrupt/' + test_ids[i] + '.png').convert('RGB')
    T_np[i] = np.asarray(im3)
print(T_np.shape)

In [ ]:
# mse if we just submitted the corrupt image as-is
x = C_np[0].astype(np.float32)
y = K_np[0].astype(np.float32)
print(((x - y) ** 2).mean())

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

STEPS = 30000
if os.environ.get('STEPS'):
    STEPS = int(os.environ['STEPS'])
SEED = 0
if os.environ.get('SEED'):
    SEED = int(os.environ['SEED'])
TIME_BUDGET_H = 10.5
if os.environ.get('TIME_BUDGET_H'):
    TIME_BUDGET_H = float(os.environ['TIME_BUDGET_H'])
dev = 'cuda'
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
print(torch.cuda.get_device_name(0))

In [ ]:
# bf16 on newer gpus, fp16 on t4, fp32 on p100
cap = torch.cuda.get_device_capability()
if cap[0] >= 8:
    AMP = True
    ADT = torch.bfloat16
elif cap[0] == 7:
    AMP = True
    ADT = torch.float16
else:
    AMP = False
    ADT = torch.float32
print(cap)
print(ADT)
scaler = torch.amp.GradScaler('cuda', enabled=(ADT is torch.float16))

## Model

Small head predicts the rotation angle, the image gets de-rotated, then a unet
cleans it up. The unet also sees the original image so it still works when the
angle is wrong or the image was never rotated.

In [ ]:
class Block(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.n1 = nn.GroupNorm(8, ci)
        self.c1 = nn.Conv2d(ci, co, 3, 1, 1)
        self.n2 = nn.GroupNorm(8, co)
        self.c2 = nn.Conv2d(co, co, 3, 1, 1)
        if ci != co:
            self.skip = nn.Conv2d(ci, co, 1)
        else:
            self.skip = nn.Identity()
        nn.init.zeros_(self.c2.weight)
        nn.init.zeros_(self.c2.bias)

    def forward(self, x):
        h = self.c1(F.silu(self.n1(x)))
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)


class Attn(nn.Module):
    def __init__(self, c, heads=4):
        super().__init__()
        self.n = nn.GroupNorm(8, c)
        self.qkv = nn.Conv2d(c, c * 3, 1)
        self.proj = nn.Conv2d(c, c, 1)
        self.h = heads
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        b, c, H, W = x.shape
        q, k, v = self.qkv(self.n(x)).reshape(b, 3, self.h, c // self.h, H * W).unbind(1)
        o = F.scaled_dot_product_attention(q.transpose(-1, -2), k.transpose(-1, -2),
                                           v.transpose(-1, -2))
        return x + self.proj(o.transpose(-1, -2).reshape(b, c, H, W))


class AngleHead(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(4, c, 3, 1, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, c, 3, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, c * 2, 3, 1, 1), nn.GroupNorm(8, c * 2), nn.SiLU(),
            nn.Conv2d(c * 2, c * 2, 3, 2, 1), nn.GroupNorm(8, c * 2), nn.SiLU(),
            nn.Conv2d(c * 2, c * 2, 3, 1, 1), nn.GroupNorm(8, c * 2), nn.SiLU(),
        )
        self.fc = nn.Linear(c * 4, 1)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        h = self.net(x)
        h = torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1)
        return self.fc(h).squeeze(1) * 15.0


class UNet(nn.Module):
    def __init__(self, cin=8, w=(128, 256, 512), nb=2):
        super().__init__()
        w1, w2, w3 = w
        self.stem = nn.Conv2d(cin, w1, 3, 1, 1)
        self.e1 = nn.ModuleList([Block(w1, w1) for _ in range(nb)])
        self.d1 = nn.Conv2d(w1, w2, 3, 2, 1)
        self.e2 = nn.ModuleList([Block(w2, w2) for _ in range(nb)])
        self.a2 = Attn(w2)
        self.d2 = nn.Conv2d(w2, w3, 3, 2, 1)
        self.mid = nn.ModuleList([Block(w3, w3), Attn(w3), Block(w3, w3)])
        self.u2 = nn.Conv2d(w3, w2, 3, 1, 1)
        self.f2 = nn.ModuleList([Block(w2 * 2, w2)] + [Block(w2, w2) for _ in range(nb - 1)])
        self.b2 = Attn(w2)
        self.u1 = nn.Conv2d(w2, w1, 3, 1, 1)
        self.f1 = nn.ModuleList([Block(w1 * 2, w1)] + [Block(w1, w1) for _ in range(nb - 1)])
        self.out_n = nn.GroupNorm(8, w1)
        self.out = nn.Conv2d(w1, 3, 3, 1, 1)
        # zero init so the model starts out as the identity
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x):
        h1 = self.stem(x)
        for b in self.e1:
            h1 = b(h1)
        h2 = self.d1(h1)
        for b in self.e2:
            h2 = b(h2)
        h2 = self.a2(h2)
        h3 = self.d2(h2)
        for m in self.mid:
            h3 = m(h3)
        g2 = self.u2(F.interpolate(h3, scale_factor=2, mode='nearest'))
        g2 = torch.cat([g2, h2], 1)
        for b in self.f2:
            g2 = b(g2)
        g2 = self.b2(g2)
        g1 = self.u1(F.interpolate(g2, scale_factor=2, mode='nearest'))
        g1 = torch.cat([g1, h1], 1)
        for b in self.f1:
            g1 = b(g1)
        return self.out(F.silu(self.out_n(g1)))


class Restorer(nn.Module):
    def __init__(self):
        super().__init__()
        self.angle = AngleHead()
        self.unet = UNet(8, (128, 256, 512), 2)

    def forward(self, x):
        # black corners show where the image was rotated
        blk = (x.sum(1, keepdim=True) < 0.012).float()
        th = self.angle(torch.cat([x, blk], 1))
        t = -th.detach() * math.pi / 180.0
        cos = torch.cos(t)
        sin = torch.sin(t)
        mat = torch.zeros(x.shape[0], 2, 3, device=x.device, dtype=x.dtype)
        mat[:, 0, 0] = cos
        mat[:, 0, 1] = -sin
        mat[:, 1, 0] = sin
        mat[:, 1, 1] = cos
        grid = F.affine_grid(mat, x.shape, align_corners=False)
        wx = F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        ones = torch.ones(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
        val = F.grid_sample(ones, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        # give the unet both the de-rotated and the original image
        inp = torch.cat([x, blk, wx, val], 1)
        return (x + self.unet(inp)).clamp(0, 1), th

In [ ]:
K = torch.from_numpy(K_np).to(dev).permute(0, 3, 1, 2).contiguous()
C = torch.from_numpy(C_np).to(dev).permute(0, 3, 1, 2).contiguous()
N = K.shape[0]
# fixed val split so different runs can be compared
g = torch.Generator().manual_seed(1234)
perm = torch.randperm(N, generator=g)
val_idx = perm[:3000].to(dev)
tr_idx = perm[3000:].to(dev)
vC = C[val_idx].float() / 255.
vK = K[val_idx].float() / 255.
v_rot = ((vC.sum(1) == 0).float().mean((1, 2)) > 0.02)
print(vC.shape)
print((((vC - vK) * 255) ** 2).mean().item())
print(v_rot.float().mean().item())

In [ ]:
BS = 256
if os.environ.get('BS'):
    BS = int(os.environ['BS'])
EVB = 500
if os.environ.get('EVB'):
    EVB = int(os.environ['EVB'])
LR = 2e-3
EMA_D = 0.9995
SYNTH = 0.5
net = Restorer().to(dev).to(memory_format=torch.channels_last)
ema = Restorer().to(dev).to(memory_format=torch.channels_last)
ema.load_state_dict(net.state_dict())
for q in ema.parameters():
    q.requires_grad_(False)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
temp = 0
for q in net.parameters():
    temp = temp + q.numel()
print(temp)

In [ ]:
X_test = torch.from_numpy(T_np).to(dev).permute(0, 3, 1, 2).float() / 255.
print(X_test.shape)

In [ ]:
n_syn = int(BS * SYNTH)
n_real = BS - n_syn
best = 1e9
t0 = time.time()
deadline = t0 + TIME_BUDGET_H * 3600
last_sub = 0.0
H = 32
W = 32
print(n_real, n_syn, STEPS)

for it in range(1, STEPS + 1):
    if it < 500:
        lr = LR * it / 500
    else:
        t = (it - 500) / max(1, STEPS - 500)
        lr = LR * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * t)))
    for gp in opt.param_groups:
        gp['lr'] = lr

    ri = tr_idx[torch.randint(len(tr_idx), (n_real,), device=dev)]
    xr = C[ri].float() / 255.
    yr = K[ri].float() / 255.
    si = tr_idx[torch.randint(len(tr_idx), (n_syn,), device=dev)]
    ys = K[si].float() / 255.

    # make our own corrupted images from the clean ones, half the batch
    with torch.no_grad():
        b = n_syn
        xs = ys
        keep = (torch.rand(b, device=dev) < 0.03)
        s = torch.exp(torch.rand(b, device=dev) * (math.log(0.72) - math.log(0.085)) + math.log(0.085))
        s = s * (~keep).float()

        m = ((torch.rand(b, device=dev) < 0.42) & ~keep).float()
        gain = 1 + m * s * (torch.rand(b, device=dev) * 0.70 - 0.45)
        bias = m * s * (torch.rand(b, device=dev) * 0.36 - 0.08)
        gam = 1 + m * s * (torch.rand(b, device=dev) * 0.95 - 0.4)
        xs = (xs.clamp(0, 1) ** gam[:, None, None, None]) * gain[:, None, None, None] + bias[:, None, None, None]
        xs = xs.clamp(0, 1)

        m = (torch.rand(b, device=dev) < 0.18) & ~keep
        dh = s * (torch.rand(b, device=dev) * 1.4 - 0.7)
        ds = 1 + s * (torch.rand(b, device=dev) * 1.4 - 0.7)
        r = xs[:, 0]
        gg = xs[:, 1]
        bl = xs[:, 2]
        yy = 0.299 * r + 0.587 * gg + 0.114 * bl
        ii = 0.596 * r - 0.274 * gg - 0.322 * bl
        qq = 0.211 * r - 0.523 * gg + 0.312 * bl
        cc = torch.cos(dh)[:, None, None]
        ss = torch.sin(dh)[:, None, None]
        sc = ds[:, None, None]
        i2 = (ii * cc - qq * ss) * sc
        q2 = (ii * ss + qq * cc) * sc
        hsv = torch.stack([yy + 0.956 * i2 + 0.621 * q2,
                           yy - 0.272 * i2 - 0.647 * q2,
                           yy - 1.106 * i2 + 1.703 * q2], 1)
        xs = torch.where(m[:, None, None, None], hsv, xs).clamp(0, 1)

        # rotation. biggest single source of error in the real data
        do_rot = (torch.rand(b, device=dev) < 0.235) & ~keep
        sign = torch.where((torch.rand(b, device=dev) < 0.5), 1.0, -1.0)
        ts = sign * (torch.rand(b, device=dev) * 11.5 + 3.0) * do_rot.float()
        t2 = ts * math.pi / 180.0
        mat = torch.zeros(b, 2, 3, device=dev, dtype=xs.dtype)
        mat[:, 0, 0] = torch.cos(t2)
        mat[:, 0, 1] = -torch.sin(t2)
        mat[:, 1, 0] = torch.sin(t2)
        mat[:, 1, 1] = torch.cos(t2)
        grid = F.affine_grid(mat, xs.shape, align_corners=False)
        xs = F.grid_sample(xs, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        ones = torch.ones(b, 1, H, W, device=dev, dtype=xs.dtype)
        valid = F.grid_sample(ones, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
        valid = (valid > 0.999).float()

        m = (torch.rand(b, device=dev) < 0.30) & ~keep
        sigma = 0.35 + 1.05 * s
        rr = torch.arange(-2, 3, device=dev, dtype=xs.dtype)
        kk = torch.exp(-(rr[None, :] ** 2) / (2 * sigma[:, None] ** 2 + 1e-8))
        kk = kk / kk.sum(1, keepdim=True)
        xk = kk.view(b, 1, 1, 5).expand(b, 3, 1, 5).reshape(b * 3, 1, 1, 5)
        yk = kk.view(b, 1, 5, 1).expand(b, 3, 5, 1).reshape(b * 3, 1, 5, 1)
        v = xs.reshape(1, b * 3, H, W)
        v = F.conv2d(F.pad(v, (2, 2, 0, 0), mode='reflect'), xk, groups=b * 3)
        v = F.conv2d(F.pad(v, (0, 0, 2, 2), mode='reflect'), yk, groups=b * 3)
        xb = v.reshape(xs.shape)
        xs = torch.where(m[:, None, None, None], xb, xs)

        m = (torch.rand(b, device=dev) < 0.12) & ~keep & (s > 0.45)
        xd = F.interpolate(F.interpolate(xs, scale_factor=0.5, mode='bilinear', align_corners=False),
                           size=(32, 32), mode='bilinear', align_corners=False)
        xs = torch.where(m[:, None, None, None], xd, xs)

        active = (torch.rand(b, device=dev) < 0.11) & ~keep
        for j in range(5):
            draw = active & (torch.rand(b, device=dev) < (0.25 + 0.6 * s))
            span = (2 + 8 * s).long().clamp(min=3)
            hh = (torch.rand(b, device=dev) * (span - 2)).long() + 2
            ww = (torch.rand(b, device=dev) * (span - 2)).long() + 2
            y0 = (torch.rand(b, device=dev) * (H - hh).clamp(min=1)).long()
            x0 = (torch.rand(b, device=dev) * (W - ww).clamp(min=1)).long()
            yy2 = torch.arange(H, device=dev)[None, :, None]
            xx2 = torch.arange(W, device=dev)[None, None, :]
            mm = ((yy2 >= y0[:, None, None]) & (yy2 < (y0 + hh)[:, None, None]) &
                  (xx2 >= x0[:, None, None]) & (xx2 < (x0 + ww)[:, None, None]))
            mm = mm & draw[:, None, None]
            col = torch.rand(b, 3, 1, 1, device=dev)
            kind = torch.rand(b, 1, 1, 1, device=dev)
            col = torch.where(kind < 0.25, torch.zeros_like(col), col)
            col = torch.where((kind >= 0.25) & (kind < 0.4),
                              torch.rand(b, 1, 1, 1, device=dev).expand(-1, 3, -1, -1), col)
            xs = torch.where(mm[:, None], col, xs)

        active = (torch.rand(b, device=dev) < 0.04) & ~keep
        for j in range(3):
            draw = active & (torch.rand(b, device=dev) < 0.5)
            y0 = torch.randint(0, H - 1, (b,), device=dev)
            th2 = torch.randint(1, 3, (b,), device=dev)
            yy3 = torch.arange(H, device=dev)[None, :]
            mm = ((yy3 >= y0[:, None]) & (yy3 < (y0 + th2)[:, None])) & draw[:, None]
            shift = int(torch.randint(2, 7, (1,)).item())
            rolled = torch.roll(xs, shift, dims=-1)
            xs = torch.where(mm[:, None, :, None], rolled, xs)

        # multiply by valid so the black corners stay exactly 0 like the real data
        m = ((torch.rand(b, device=dev) < 0.45) & ~keep).float()
        xs = xs + torch.randn_like(xs) * (m * s * 0.11)[:, None, None, None] * valid
        xs = xs + torch.randn_like(xs) * ((~keep).float() * (0.004 + 0.010 * s))[:, None, None, None] * valid

        dens = s * 0.055
        active = (torch.rand(b, device=dev) < 0.12) & ~keep
        hit = (torch.rand(b, 1, H, W, device=dev) < dens[:, None, None, None]) & active[:, None, None, None]
        col = torch.rand(b, 3, H, W, device=dev)
        col = torch.where(torch.rand(b, 3, 1, 1, device=dev) < 0.35, (col > 0.5).float(), col)
        xs = torch.where(hit, col, xs)

        xs = xs.clamp(0, 1)
        xs = torch.where(keep[:, None, None, None], ys, xs)
        ts = torch.where(keep, torch.zeros_like(ts), ts)
        xs = (xs * 255).round() / 255.0

    x = torch.cat([xr, xs])
    y = torch.cat([yr, ys])
    if torch.rand(1).item() < 0.5:
        x = x.flip(-1)
        y = y.flip(-1)
        ts = -ts
    x = x.to(memory_format=torch.channels_last)

    with torch.autocast('cuda', ADT, enabled=AMP):
        pred, th = net(x)
        loss_mse = F.mse_loss(pred.float(), y)
        loss_ang = F.smooth_l1_loss(th[n_real:].float(), ts, beta=1.0)
        # angle loss only on our synthetic half where we know the true angle
        loss = loss_mse + 0.02 * loss_ang
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()

    d = min(EMA_D, (it + 1) / (it + 10))
    with torch.no_grad():
        for pe, pn in zip(ema.parameters(), net.parameters()):
            pe.lerp_(pn.detach(), 1 - d)
        for be, bn in zip(ema.buffers(), net.buffers()):
            be.copy_(bn)

    if it % 500 == 0 or it == STEPS:
        ema.eval()
        tot = []
        for i in range(0, len(vC), EVB):
            with torch.no_grad():
                with torch.autocast('cuda', ADT, enabled=AMP):
                    pr, _ = ema(vC[i:i + EVB].to(memory_format=torch.channels_last))
            pr = (pr.float() * 255).round().clamp(0, 255) / 255.
            tot.append((((pr - vK[i:i + EVB]) * 255) ** 2).mean((1, 2, 3)))
        ema.train()
        per = torch.cat(tot)
        m_all = per.mean().item()
        m_rot = per[v_rot].mean().item()
        m_non = per[~v_rot].mean().item()
        print(it, round(time.time() - t0), round(m_all, 2), round(m_rot, 2), round(m_non, 2))
        if m_all < best:
            best = m_all
            torch.save({'ema': ema.state_dict(), 'val': m_all, 'step': it, 'seed': SEED}, out + 'ckpt.pt')
            # write a submission now and then, in case the session gets killed
            if time.time() - last_sub > 1500:
                ema.eval()
                acc = torch.zeros_like(X_test)
                for i in range(0, len(X_test), EVB):
                    with torch.no_grad():
                        with torch.autocast('cuda', ADT, enabled=AMP):
                            pr, _ = ema(X_test[i:i + EVB])
                    acc[i:i + EVB] = acc[i:i + EVB] + pr.float()
                for i in range(0, len(X_test), EVB):
                    with torch.no_grad():
                        with torch.autocast('cuda', ADT, enabled=AMP):
                            pr, _ = ema(X_test[i:i + EVB].flip(-1))
                    acc[i:i + EVB] = acc[i:i + EVB] + pr.float().flip(-1)
                ema.train()
                flat = ((acc / 2).clamp(0, 1) * 255).round().clamp(0, 255).byte()
                flat = flat.permute(0, 2, 3, 1).reshape(len(X_test), -1).cpu().numpy()
                cols = ['pixel_' + str(i) for i in range(3072)]
                fo = open(out + 'submission.csv.tmp', 'w')
                fo.write('id,' + ','.join(cols) + '\n')
                for i in range(len(test_ids)):
                    fo.write(test_ids[i] + ',' + ','.join(map(str, flat[i].tolist())) + '\n')
                fo.close()
                os.replace(out + 'submission.csv.tmp', out + 'submission.csv')
                last_sub = time.time()
                print('saved submission')
    # kaggle kills the session at 12h, stop before that
    if time.time() > deadline:
        print('out of time', it)
        break
print(best)

## Submission

Predict the test set twice, normal and horizontally flipped, and average.
Pixels are flattened as height x width x channel.

In [ ]:
ck = torch.load(out + 'ckpt.pt', map_location=dev, weights_only=False)
net2 = Restorer().to(dev)
net2.load_state_dict(ck['ema'])
net2.eval()
print(ck['val'], ck['step'])

In [ ]:
# average the normal and flipped predictions
acc = torch.zeros_like(X_test)
for i in range(0, len(X_test), EVB):
    with torch.no_grad():
        with torch.autocast('cuda', ADT, enabled=AMP):
            pr, _ = net2(X_test[i:i + EVB])
    acc[i:i + EVB] = acc[i:i + EVB] + pr.float()
for i in range(0, len(X_test), EVB):
    with torch.no_grad():
        with torch.autocast('cuda', ADT, enabled=AMP):
            pr, _ = net2(X_test[i:i + EVB].flip(-1))
    acc[i:i + EVB] = acc[i:i + EVB] + pr.float().flip(-1)
result = (acc / 2).clamp(0, 1)
print(result.shape)

In [ ]:
flat = (result * 255).round().clamp(0, 255).byte().permute(0, 2, 3, 1).reshape(len(X_test), -1).cpu().numpy()
print(flat.shape)
cols = ['pixel_' + str(i) for i in range(3072)]
fo = open(out + 'submission.csv.tmp', 'w')
fo.write('id,' + ','.join(cols) + '\n')
for i in range(len(test_ids)):
    fo.write(test_ids[i] + ',' + ','.join(map(str, flat[i].tolist())) + '\n')
fo.close()
os.replace(out + 'submission.csv.tmp', out + 'submission.csv')
print('done')

In [ ]:
# check shape
f3 = open(out + 'submission.csv')
temp = []
for r in csv.reader(f3):
    temp.append(r)
f3.close()
print(len(temp))
print(len(temp[0]))
print(temp[1][:5])